# Matrices TF, IDF y TF-IDF

Se procesan 100 documentos locales de Project Gutenberg. Las filas representan documentos y las columnas, terminos. Se emplea frecuencia absoluta, sin eliminar stopwords ni aplicar stemming o lematizacion.

Requisitos: Python con `numpy`, `pandas`, `scipy`, `IPython` y Jupyter. La ruta del corpus puede ajustarse en la siguiente celda.

In [1]:
from pathlib import Path
from collections import Counter
import hashlib
import re
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from IPython.display import display

RUTA_CORPUS = Path(r"C:\Users\Jorge\Documents\EPN-7mo Semestre\Recuperacion Informacion\epub")
N_OBJETIVO = 100
PATRON_TOKEN = re.compile(r"[^\W_]+", flags=re.UNICODE)

def tokenizar(texto):
    return PATRON_TOKEN.findall(texto.casefold())

assert tokenizar("Tutorías, TUTORÍAS; año 2026.") == ["tutorías", "tutorías", "año", "2026"]
if not RUTA_CORPUS.is_dir():
    raise FileNotFoundError(f"No existe el corpus: {RUTA_CORPUS}")

## 1. Carga y preprocesamiento

Se buscan TXT en todas las subcarpetas y se ordenan por identificador numerico y ruta. Se toman los primeros 100 archivos validos con identificadores y contenido diferentes; asi la seleccion es reproducible mientras el corpus no cambie. Se lee UTF-8 con BOM opcional y, si falla, Windows-1252. Se exige una cabecera de Gutenberg y contenido no vacio. La huella SHA-256 del texto completo permite descartar copias exactas, aunque no identifica todas las ediciones similares.

Se procesa el TXT completo, incluidos encabezados y licencia, como en el cuaderno anterior. `casefold()` convierte a minusculas; la puntuacion separa tokens y se conservan letras y numeros Unicode, incluidas tildes y ñ. No se aplica normalizacion Unicode adicional.

In [2]:
archivos = sorted(RUTA_CORPUS.rglob("*.txt"),
                  key=lambda p: (int(p.parent.name) if p.parent.name.isdigit() else float("inf"),
                                 str(p.relative_to(RUTA_CORPUS)).casefold()))
registros = []
contadores = []
ids_vistos = set()
huellas = set()
for archivo in archivos:
    coincidencia = re.fullmatch(r"pg(\d+)", archivo.stem, re.IGNORECASE)
    if not coincidencia:
        continue
    doc_id = coincidencia.group(1)
    if doc_id in ids_vistos:
        continue
    try:
        try:
            texto = archivo.read_text(encoding="utf-8-sig")
            codificacion = "utf-8-sig"
        except UnicodeDecodeError:
            texto = archivo.read_text(encoding="cp1252")
            codificacion = "cp1252"
    except (OSError, UnicodeError):
        continue
    if "project gutenberg" not in texto[:5000].casefold():
        continue
    huella = hashlib.sha256(texto.encode("utf-8")).hexdigest()
    if huella in huellas:
        continue
    frecuencias = Counter(tokenizar(texto))
    if not frecuencias:
        continue
    ids_vistos.add(doc_id)
    huellas.add(huella)
    contadores.append(frecuencias)
    registros.append({"ID Gutenberg": doc_id, "Archivo": str(archivo.relative_to(RUTA_CORPUS)),
                      "Codificacion": codificacion, "Tokens": sum(frecuencias.values()),
                      "Terminos diferentes": len(frecuencias)})
    if len(registros) == N_OBJETIVO:
        break
if len(registros) != N_OBJETIVO:
    raise ValueError(f"Solo se encontraron {len(registros)} documentos validos diferentes")
resumen = pd.DataFrame(registros)
ids = resumen["ID Gutenberg"].tolist()
N = len(ids)
with pd.option_context("display.max_rows", 100):
    display(resumen)
print(f"Documentos: {N} | Tokens totales: {resumen['Tokens'].sum():,}")

,ID Gutenberg,Archivo,Codificacion,Tokens,Terminos diferentes
0,1,1\pg1.txt,utf-8-sig,4930,1212
1,2,2\pg2.txt,utf-8-sig,3657,844
2,10,10\pg10.txt,utf-8-sig,857542,13061
3,11,11\pg11.txt,utf-8-sig,30588,3031
4,12,12\pg12.txt,utf-8-sig,34272,3284
5,13,13\pg13.txt,utf-8-sig,8474,1978
6,14,14\pg14.txt,utf-8-sig,288288,18882
7,15,15\pg15.txt,utf-8-sig,222537,17347
8,16,16\pg16.txt,utf-8-sig,51199,5218
9,17,17\pg17.txt,utf-8-sig,286169,6004


Documentos: 100 | Tokens totales: 13,156,915


## 2. Matriz TF

Cada valor es el numero de apariciones de un termino en un documento. La matriz completa se conserva en formato disperso CSR. La tabla muestra 10 documentos y 20 terminos seleccionados por su frecuencia documental.

In [3]:
vocabulario = sorted({termino for frecuencias in contadores for termino in frecuencias})
columnas = {termino: i for i, termino in enumerate(vocabulario)}
filas, indices, valores = [], [], []
for fila, frecuencias in enumerate(contadores):
    for termino, frecuencia in frecuencias.items():
        filas.append(fila)
        indices.append(columnas[termino])
        valores.append(frecuencia)
matriz_tf = csr_matrix((valores, (filas, indices)), shape=(N, len(vocabulario)), dtype=np.int64)
del filas, indices, valores
print(f"Vocabulario: {len(vocabulario):,}")
print(f"Dimensiones TF: {matriz_tf.shape}; valores distintos de cero: {matriz_tf.nnz:,}")

# Se seleccionan terminos frecuentes, intermedios y raros presentes en la vista.
df = np.asarray((matriz_tf > 0).sum(axis=0)).ravel()
candidatos = np.flatnonzero(np.asarray((matriz_tf[:10] > 0).sum(axis=0)).ravel())
orden = sorted(candidatos, key=lambda i: (-df[i], vocabulario[i]))
frecuentes = orden[:7]
restantes = [i for i in orden if i not in frecuentes]
intermedios = sorted(restantes, key=lambda i: (abs(int(df[i]) - N / 2), vocabulario[i]))[:7]
restantes = [i for i in restantes if i not in intermedios]
raros = sorted(restantes, key=lambda i: (df[i], vocabulario[i]))[:6]
columnas_vista = frecuentes + intermedios + raros
terminos_vista = [vocabulario[i] for i in columnas_vista]
vista_tf = pd.DataFrame(matriz_tf[:10, columnas_vista].toarray(), index=ids[:10], columns=terminos_vista)
vista_tf.index.name = "ID Gutenberg"
display(vista_tf)

Vocabulario: 191,063
Dimensiones TF: (100, 191063); valores distintos de cero: 881,166


,000,07042,1,2,20,2001,3,abruptly,admire,ambition,anxiously,apple,application,bind,0006,0022,00620,0064,0086,0107
ID Gutenberg,,,,,,,,,,,,,,,,,,,,
1,2,1,53,7,1,3,10,0,0,0,0,0,0,0,0,0,0,0,0,0
2,1,1,48,5,1,1,10,0,0,0,0,0,0,0,0,0,0,0,0,0
10,1,1,2831,2724,1466,1,2570,0,0,0,0,8,0,49,0,0,0,0,0,0
11,1,1,47,3,1,1,11,0,0,1,14,1,0,0,0,0,0,0,0,0
12,1,1,51,5,1,1,12,0,1,0,8,0,0,0,0,0,0,0,0,0
13,1,1,48,4,1,1,11,0,0,1,0,0,0,0,0,0,0,0,0,0
14,2135,1,4893,2388,401,2,1839,2,0,0,0,0,1,0,1,1,1,5,1,10
15,20,1,53,6,5,1,16,2,3,3,0,4,6,2,0,0,0,0,0,0
16,1,1,48,4,1,1,11,2,2,1,2,0,0,0,0,0,0,0,0,0


## 3. Vector IDF

DF cuenta documentos que contienen el termino, no apariciones. Para cada columna se calcula exactamente $IDF(t)=\ln(N/df(t))+1$, mediante el logaritmo natural de NumPy. IDF es un vector de un valor por termino; una matriz IDF de las mismas dimensiones que TF repetiria ese vector en cada fila, sin necesidad de almacenarla.

In [4]:
idf = np.log(N / df) + 1
tabla_idf = pd.DataFrame({"Termino": terminos_vista, "DF": df[columnas_vista],
                          "IDF": idf[columnas_vista],
                          "Grupo": ["Frecuente"] * 7 + ["Intermedio"] * 7 + ["Poco frecuente"] * 6})
print(f"Dimensiones IDF: {idf.shape} (vector); matriz repetida conceptual: {matriz_tf.shape}")
display(tabla_idf)

Dimensiones IDF: (191063,) (vector); matriz repetida conceptual: (100, 191063)


,Termino,DF,IDF,Grupo
0,000,100,1.000000,Frecuente
1,07042,100,1.000000,Frecuente
2,1,100,1.000000,Frecuente
3,2,100,1.000000,Frecuente
4,20,100,1.000000,Frecuente
5,2001,100,1.000000,Frecuente
6,3,100,1.000000,Frecuente
7,abruptly,50,1.693147,Intermedio
8,admire,50,1.693147,Intermedio
9,ambition,50,1.693147,Intermedio


## 4. Matriz TF-IDF

Cada valor es TF multiplicado por el IDF de la misma columna. No se aplica normalizacion L1 ni L2. La vista usa los mismos documentos y terminos que TF e IDF.

In [5]:
matriz_tfidf = matriz_tf.multiply(idf).tocsr()
print(f"Dimensiones TF-IDF: {matriz_tfidf.shape}; valores distintos de cero: {matriz_tfidf.nnz:,}")
vista_tfidf = pd.DataFrame(matriz_tfidf[:10, columnas_vista].toarray(), index=ids[:10], columns=terminos_vista)
vista_tfidf.index.name = "ID Gutenberg"
display(vista_tfidf.round(4))

# Se muestran calculos con frecuencias reales.
ejemplos = []
for j in [columnas_vista[0], columnas_vista[7], columnas_vista[14]]:
    fila = int(matriz_tf[:, j].nonzero()[0][0])
    ejemplos.append({"ID Gutenberg": ids[fila], "Termino": vocabulario[j],
                     "TF": int(matriz_tf[fila, j]), "DF": int(df[j]), "IDF": idf[j],
                     "TF x IDF": matriz_tfidf[fila, j]})
display(pd.DataFrame(ejemplos))

Dimensiones TF-IDF: (100, 191063); valores distintos de cero: 881,166


,000,07042,1,2,20,2001,3,abruptly,admire,ambition,anxiously,apple,application,bind,0006,0022,00620,0064,0086,0107
ID Gutenberg,,,,,,,,,,,,,,,,,,,,
1,2.0,1.0,53.0,7.0,1.0,3.0,10.0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2,1.0,1.0,48.0,5.0,1.0,1.0,10.0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
10,1.0,1.0,2831.0,2724.0,1466.0,1.0,2570.0,0.0000,0.0000,0.0000,0.0000,13.5452,0.0000,82.9642,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
11,1.0,1.0,47.0,3.0,1.0,1.0,11.0,0.0000,0.0000,1.6931,23.7041,1.6931,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
12,1.0,1.0,51.0,5.0,1.0,1.0,12.0,0.0000,1.6931,0.0000,13.5452,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
13,1.0,1.0,48.0,4.0,1.0,1.0,11.0,0.0000,0.0000,1.6931,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
14,2135.0,1.0,4893.0,2388.0,401.0,2.0,1839.0,3.3863,0.0000,0.0000,0.0000,0.0000,1.6931,0.0000,5.6052,5.6052,5.6052,28.0259,5.6052,56.0517
15,20.0,1.0,53.0,6.0,5.0,1.0,16.0,3.3863,5.0794,5.0794,0.0000,6.7726,10.1589,3.3863,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
16,1.0,1.0,48.0,4.0,1.0,1.0,11.0,3.3863,3.3863,1.6931,3.3863,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


,ID Gutenberg,Termino,TF,DF,IDF,TF x IDF
0,1,000,2,100,1.000000,2.000000
1,14,abruptly,2,50,1.693147,3.386294
2,14,0006,1,1,5.605170,5.605170


## 5. Comprobaciones

Se comprueban las dimensiones, las frecuencias, DF y la formula para todas las columnas. Tambien se comprueba el producto completo en las entradas no nulas y la coincidencia del patron de ceros. Los ejemplos anteriores permiten revisar tres calculos individuales.

In [6]:
assert N == 100 and len(set(ids)) == N and len(huellas) == N
assert matriz_tf.shape == (N, len(vocabulario)) and len(columnas) == len(vocabulario)
assert all(columnas[t] == i for i, t in enumerate(vocabulario))
assert np.issubdtype(matriz_tf.dtype, np.integer) and np.all(matriz_tf.data > 0)
assert np.array_equal(np.asarray(matriz_tf.sum(axis=1)).ravel(), resumen["Tokens"].to_numpy())
df_independiente = Counter(t for frecuencias in contadores for t in frecuencias)
assert np.array_equal(df, np.array([df_independiente[t] for t in vocabulario]))
assert np.all((df >= 1) & (df <= N))
assert np.allclose(idf, np.log(N / df) + 1)
assert np.isclose(np.log(N / N) + 1, 1)
assert np.isclose(np.log(N / 1) + 1, np.log(100) + 1)
assert np.allclose(idf[df == N], 1)
assert np.allclose(idf[df == 1], np.log(100) + 1)
assert matriz_tf.shape == matriz_tfidf.shape
assert np.array_equal(matriz_tf.indptr, matriz_tfidf.indptr)
assert np.array_equal(matriz_tf.indices, matriz_tfidf.indices)
assert np.allclose(matriz_tfidf.data, matriz_tf.data * idf[matriz_tf.indices])
for ejemplo in ejemplos:
    fila = ids.index(ejemplo["ID Gutenberg"])
    termino = ejemplo["Termino"]
    assert matriz_tf[fila, columnas[termino]] == contadores[fila][termino]
    assert np.isclose(ejemplo["TF x IDF"], ejemplo["TF"] * ejemplo["IDF"])
print("Todas las verificaciones fueron correctas.")

Todas las verificaciones fueron correctas.
